Лабораторна робота №2

Частина 1.

1) Встановила python, створила віртуальне середовище, активувавши яке, завантажила необхідні бібліотеки (pandas, numpy, jupyter, seaborn, matplotlib).

2) Створила функцію для завантаження файлів, що містять значення VHI-індексів для кожної з адміністративних одиниць України. Додала умову, за якої файли, які були вже завантажені раніше, не будуть довантажуватись знову при повторному запуску скрипта. 

In [1]:
import urllib.request
import os
from datetime import datetime

def download_vhi_data(folder="province_data"): # завантажуємо дані VHI для всіх областей України (1-27), якщо вони ще не завантажені
    os.makedirs(folder, exist_ok=True)
    province_ids = range(1, 28)
    url_template = "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={id}&year1=1981&year2=2024&type=Mean"

    print(f"Дані завантажуються в папку '{folder}' ")
    
    for pid in province_ids:
        # перевіряємо, чи файл для цієї області вже був колись завантажений  
        existing_files = [f for f in os.listdir(folder) if f.startswith(f"VHI_province{pid}_") and f.endswith(".txt")]

        #якщо так - пропускаємо
        if existing_files:
            print(f"Область {pid}: Файл вже існує ({existing_files[0]}).")
            continue
        
        # якщо файлу немає — завантажуємо
        url = url_template.format(id=pid)
        now = datetime.now().strftime("%d.%m.%Y_%H:%M:%S")
        filename = f"{folder}/VHI_province{pid}_{now}.txt" # До назви файлу додаю дату та час зберігання
        
        try:
            urllib.request.urlretrieve(url, filename)
            print(f"Область {pid}: Успішно завантажено як {filename}")
        except Exception as e:
            print(f"Область {pid}: Помилка при завантаженні! {e}")
            
    print("Файли для усіх областей успішно завантажились")



In [2]:
# Виклик функції
download_vhi_data()

Дані завантажуються в папку 'province_data' 
Область 1: Файл вже існує (VHI_province1_07.03.2026_16:59:12.txt).
Область 2: Файл вже існує (VHI_province2_07.03.2026_16:59:19.txt).
Область 3: Файл вже існує (VHI_province3_07.03.2026_16:59:21.txt).
Область 4: Файл вже існує (VHI_province4_07.03.2026_16:59:23.txt).
Область 5: Файл вже існує (VHI_province5_07.03.2026_16:59:24.txt).
Область 6: Файл вже існує (VHI_province6_07.03.2026_16:59:26.txt).
Область 7: Файл вже існує (VHI_province7_07.03.2026_16:59:27.txt).
Область 8: Файл вже існує (VHI_province8_07.03.2026_16:59:28.txt).
Область 9: Файл вже існує (VHI_province9_07.03.2026_16:59:30.txt).
Область 10: Файл вже існує (VHI_province10_07.03.2026_16:59:31.txt).
Область 11: Файл вже існує (VHI_province11_07.03.2026_16:59:32.txt).
Область 12: Файл вже існує (VHI_province12_07.03.2026_16:59:34.txt).
Область 13: Файл вже існує (VHI_province13_07.03.2026_16:59:35.txt).
Область 14: Файл вже існує (VHI_province14_07.03.2026_16:59:36.txt).
Область

3) Створила функцію для data cleaning, яка читає всі текстові файли NOAA для областей України, очищає їх від HTML-тегів і зайвих колонок, перетворює дані у числовий формат, заповнює пропуски середнім значенням (щоб в майбутньому можна було робити статистику, обчислювати середнє арифметичне і тд.)
Створила функцію для об'єднання усіх областей в один датафрейм з колонками Year, Week, VHI, Province_ID та Province_Name.

In [51]:

import pandas as pd
import re
import numpy as np
import os

def clean_data(file_path, province_id):
    noaa_names = {
        1: "Cherkasy", 2: "Chernihiv", 3: "Chernivtsi", 4: "Crimea", 5: "Dnipropetrovsk",
        6: "Donetsk", 7: "Ivano-Frankivsk", 8: "Kharkiv", 9: "Kherson", 10: "Khmelnytskyy",
        11: "Kyiv", 12: "Kyiv City", 13: "Kirovohrad", 14: "Luhansk", 15: "Lviv",
        16: "Mykolayiv", 17: "Odessa", 18: "Poltava", 19: "Rivne", 20: "Sevastopol",
        21: "Sumy", 22: "Ternopil", 23: "Transcarpathia", 24: "Vinnytsya", 25: "Volyn",
        26: "Zaporizhzhya", 27: "Zhytomyr"
    } # словник індексів та назв областей, які відповідають їм за NOAA (за англійським алфавітом)
    
    data = []
    with open(file_path, 'r', encoding='utf-8') as f: # відкриваємо файл в кодуванні UTF-8 та читаємо всі рядки у список lines
        lines = f.readlines()
    
    for line in lines: # цикл, що проходиться по кожному рядку файлу
        line = line.strip() # прибираємо пробіли з початку та кінця рядка
        line = re.sub(r'<[^>]+>', '', line) # видаляємо HTML-теги (<pre>, <tt>) та пропускаємо порожні рядки
        if not line:
            continue
        
        if line.lower().startswith('year,week'): # пропускаємо заголовки
            continue
        
        if line[0].isdigit(): # якщо рядок починається з цифри
            line = line.rstrip(',')  # видаляємо кому в кінці
            parts = line.split(',') # розбиваємо рядок на список [Year, Week, SMN, SMT, VCI, TCI, VHI]
            if len(parts) >= 7: # перевіряємо, що в рядку достатньо колонок
                try:
                    year = int(parts[0].strip()) # обираємо три потрібні колонки
                    week = int(parts[1].strip())
                    vhi = float(parts[6].strip())
                    data.append([year, week, vhi])
                except:
                    continue
    
    df = pd.DataFrame(data, columns=['Year', 'Week', 'VHI']) # створюємо датафрейм
    
    df.loc[df['VHI'] == -1, 'VHI'] = np.nan # замінюємо значення -1 на NaN
    df['VHI'] = df['VHI'].fillna(df['VHI'].mean()) # заповнюємо пропуски середнім арифметичним
    
    # додаємо інформацію про область
    df['Province_ID'] = province_id
    df['Province_Name'] = noaa_names.get(province_id, "Unknown")
    
    return df

def build_dataframe(folder_path):
    all_frames = []
    files = [f for f in os.listdir(folder_path) if f.endswith(".txt")] # шукаємо в папці усі текстові файли 
    files = sorted(files, key=lambda x: int(re.search(r'province(\d+)', x).group(1))) # сортуємо по айді

    # для кожного файлу витягуємо айді з назви, викликаємо функцію для data cleaning та додаємо результат у список
    for file in files:
        match = re.search(r'province(\d+)', file)
        if match:
            province_id = int(match.group(1))
            path = os.path.join(folder_path, file)
            clean_df = clean_data(path, province_id)
            all_frames.append(clean_df)
    
    return pd.concat(all_frames, ignore_index=True) # об'єднуємо всі області в одну таблицю
    
df = build_dataframe("province_data")

In [52]:
# можна переглянути перші та останні 50 рядків

print(df.head(50))
#print(df.tail(50))

    Year  Week    VHI  Province_ID Province_Name
0   1982     1  42.23            1      Cherkasy
1   1982     2  39.29            1      Cherkasy
2   1982     3  37.68            1      Cherkasy
3   1982     4  35.00            1      Cherkasy
4   1982     5  34.06            1      Cherkasy
5   1982     6  33.01            1      Cherkasy
6   1982     7  31.65            1      Cherkasy
7   1982     8  31.40            1      Cherkasy
8   1982     9  31.73            1      Cherkasy
9   1982    10  31.44            1      Cherkasy
10  1982    11  30.80            1      Cherkasy
11  1982    12  31.89            1      Cherkasy
12  1982    13  33.37            1      Cherkasy
13  1982    14  34.20            1      Cherkasy
14  1982    15  37.78            1      Cherkasy
15  1982    16  43.81            1      Cherkasy
16  1982    17  49.85            1      Cherkasy
17  1982    18  54.97            1      Cherkasy
18  1982    19  55.86            1      Cherkasy
19  1982    20  54.5

4. Реалізувала процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), а я замінила індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька). Я це зробила шляхом формування словника перекладу та застосування алгоритму, який присвоює кожній області індекс відповідно до її позиції в українському алфавітному списку. Області тепер відсортовані за алфавітом, а міста я відправила в кінець списку. 

In [53]:
def reindex_ua(df):
    # список областей за українським алфавітом
    ua_ordered_names = [
        "Вінницька", "Волинська", "Дніпропетровська", "Донецька", "Житомирська",
        "Закарпатська", "Запорізька", "Івано-Франківська", "Київська", "Кіровоградська",
        "Луганська", "Львівська", "Миколаївська", "Одеська", "Полтавська",
        "Рівненська", "Сумська", "Тернопільська", "Харківська", "Херсонська",
        "Хмельницька", "Черкаська", "Чернівецька", "Чернігівська", "Республіка Крим",
        "м. Київ", "м. Севастополь"
    ]
    
    # словник перекладу з NOAA на UA
    translation_map = {
        "Vinnytsya": "Вінницька", "Volyn": "Волинська", "Dnipropetrovsk": "Дніпропетровська",
        "Donetsk": "Донецька", "Zhytomyr": "Житомирська", "Transcarpathia": "Закарпатська",
        "Zaporizhzhya": "Запорізька", "Ivano-Frankivsk": "Івано-Франківська", 
        "Kyiv": "Київська", "Kirovohrad": "Кіровоградська", "Luhansk": "Луганська", 
        "Lviv": "Львівська", "Mykolayiv": "Миколаївська", "Odessa": "Одеська", 
        "Poltava": "Полтавська", "Rivne": "Рівненська", "Sumy": "Сумська", 
        "Ternopil": "Тернопільська", "Kharkiv": "Харківська", "Kherson": "Херсонська", 
        "Khmelnytskyy": "Хмельницька", "Cherkasy": "Черкаська", "Chernivtsi": "Чернівецька", 
        "Chernihiv": "Чернігівська", "Crimea": "Республіка Крим", "Kyiv City": "м. Київ", 
        "Sevastopol": "м. Севастополь"
    }

    new_df = df.copy() # щоб не змінювати оригінальний датафрейм, зробимо його копію
    
    # перекладаємо назви
    new_df['Province_Name'] = new_df['Province_Name'].map(translation_map)
    
    # створюємо словник id на основі правильного українського списку
    name_to_id = {name: i + 1 for i, name in enumerate(ua_ordered_names)}
    
    # призначаємо нові id
    new_df['Province_ID'] = new_df['Province_Name'].map(name_to_id)
    
    return new_df


In [54]:
# виклик функції

df_ua = reindex_ua(df)
df_ua = df_ua.sort_values(by=['Province_ID', 'Year', 'Week']).reset_index(drop=True) # сортуємо за id, роком та тижнем

# показати перші 5 рядків
display(df_ua.head()) 

,Year,Week,VHI,Province_ID,Province_Name
0,1982,1,45.90,1,Вінницька
1,1982,2,45.34,1,Вінницька
2,1982,3,44.88,1,Вінницька
3,1982,4,41.60,1,Вінницька
4,1982,5,39.29,1,Вінницька


5. Реалізувала процедури для формування вибірок наступного виду:

1) Ряд VHI для області за вказаний рік;

In [56]:
def vhi_by_year(df, province_id, year):
    # Фільтруємо за двома умовами одночасно (за id області та за роком)
    selection = df[(df['Province_ID'] == province_id) & (df['Year'] == year)]
    
    if not selection.empty:
        province_name = selection['Province_Name'].iloc[0]
        print(f"Вибірка для області {province_name} за {year} рік:")
    else:
        print(f"Дані для ID {province_id} за {year} рік не знайдені.")
        
    # Повертаємо колонки лише з тижнем та індексом
    return selection[['Week', 'VHI']]

In [57]:
# приклад виводу для вінницької області

vhi_data = vhi_by_year(df_ua, 1, 2023)
display(vhi_data)

Вибірка для області Вінницька за 2023 рік:


,Week,VHI
2132,1,40.09
2133,2,43.61
2134,3,46.74
2135,4,48.03
2136,5,47.88
2137,6,46.98
2138,7,46.11
2139,8,46.95
2140,9,48.68
2141,10,49.84


2) Ряд VHI за вказаний діапазон років для вказаних областей;

In [58]:
def vhi_range(df, provinces_list, start_year, end_year):
    # фільтруємо одразу за областю у списку та роком у межах [start; end]
    selection = df[
        (df['Province_ID'].isin(provinces_list)) & 
        (df['Year'] >= start_year) & 
        (df['Year'] <= end_year)
    ]
    return selection

In [59]:
# приклад виклику для Вінницької та Волинської областей з 2010 по 2012: 
display(vhi_range(df_ua, [1, 2], 2010, 2012))

# як бачимо, дані виводяться цільним списком одна область за іншою

,Year,Week,VHI,Province_ID,Province_Name
1456,2010,1,52.04,1,Вінницька
1457,2010,2,51.05,1,Вінницька
1458,2010,3,52.37,1,Вінницька
1459,2010,4,52.69,1,Вінницька
1460,2010,5,53.16,1,Вінницька
...,...,...,...,...,...
3843,2012,48,48.96,2,Волинська
3844,2012,49,49.24,2,Волинська
3845,2012,50,46.05,2,Волинська
3846,2012,51,46.54,2,Волинська


3) Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани;

In [60]:
# функція для знаходження назви області за айді, щоб був зручніший вивід
def get_province_names(df, provinces_list):
    names = df[df['Province_ID'].isin(provinces_list)]['Province_Name'].unique()
    return ", ".join(names) # склеюємо через кому

# функція для знаходження екстремумів
def vhi_extremes(df, provinces_list, years_list):
    subset = df[(df['Province_ID'].isin(provinces_list)) & (df['Year'].isin(years_list))]
    if subset.empty: return "Даних не знайдено"

    names_str = get_province_names(df, provinces_list)
    v_min = subset['VHI'].min()
    v_max = subset['VHI'].max()
    
    print(f"Екстремуми VHI для області(ей) {names_str} за рік(роки) {years_list}:")
    print(f"Мінімальне значення: {v_min}")
    print(f"Максимальне значення: {v_max}")
    return v_min, v_max

# фунцкія для знаходження середнього значення
def vhi_mean(df, provinces_list, years_list):
    subset = df[(df['Province_ID'].isin(provinces_list)) & (df['Year'].isin(years_list))]
    if subset.empty: return None

    names_str = get_province_names(df, provinces_list)
    mean_value = subset['VHI'].mean()
    print(f"Середнє значення VHI для області(ей) {names_str} за рік(роки) {years_list}: {mean_value:.2f}")
    return mean_value

# функція для знаходження медіани
def vhi_median(df, provinces_list, years_list):
    subset = df[(df['Province_ID'].isin(provinces_list)) & (df['Year'].isin(years_list))]
    if subset.empty: return None

    names_str = get_province_names(df, provinces_list)
    median_value = subset['VHI'].median()
    print(f"Медіана VHI для област(ей) {names_str} за рік(роки) {years_list}: {median_value:.2f}")
    return median_value

# функція для виклику одразу усіх трьох попередніх
def analyze_vhi_stats(df, provinces, years):
    print(f"Аналіз даних для таких областей: {provinces}")
    vhi_extremes(df, provinces, years)
    vhi_mean(df, provinces, years)
    vhi_median(df, provinces, years)
    print("="*40) # це просто для краси


In [61]:
# можна викликати останню четверту функцію, яка викличе всі інші
analyze_vhi_stats(df_ua, [1, 2], [2020, 2021])

# а можна і окремо викликати
extremes = vhi_extremes(df_ua, [1], [2010, 2011])
mean = vhi_mean(df_ua, [1], [2010, 2011])
median = vhi_median(df_ua, [1], [2010, 2011])

Аналіз даних для таких областей: [1, 2]
Екстремуми VHI для області(ей) Вінницька, Волинська за рік(роки) [2020, 2021]:
Мінімальне значення: 33.55
Максимальне значення: 71.59
Середнє значення VHI для області(ей) Вінницька, Волинська за рік(роки) [2020, 2021]: 51.24
Медіана VHI для област(ей) Вінницька, Волинська за рік(роки) [2020, 2021]: 50.23
Екстремуми VHI для області(ей) Вінницька за рік(роки) [2010, 2011]:
Мінімальне значення: 36.4
Максимальне значення: 68.1
Середнє значення VHI для області(ей) Вінницька за рік(роки) [2010, 2011]: 48.45
Медіана VHI для област(ей) Вінницька за рік(роки) [2010, 2011]: 46.88
